In [ ]:
import pandas as pd

In [ ]:
# Load dataframe
clean_df = pd.read_csv(r"C:\\Code\\kickstarter_scrape\\clean_df_progress_save.csv")

In [ ]:
# number of rows
print(f"\nTotal number of rows: {clean_df.shape[0]}")


Total number of rows in the scraped subset: 183709


In [9]:
# number of projects with unique risks_html content
num_unique_risks = clean_df['risks_html'].nunique()
print(f"\nNumber of unique values in 'risks_html': {num_unique_risks}")


Number of unique values in 'risks_html': 44678


In [6]:
# number of unique values in risks_html with state 'failed'
num_unique_risks_failed = clean_df[clean_df['state'] == 'failed']['risks_html'].nunique()
print(f"Number of unique values in 'risks_html' for failed projects: {num_unique_risks_failed}")

# number of unique values in risks_html with state 'successful'
num_unique_risks_successful = clean_df[clean_df['state'] == 'successful']['risks_html'].nunique()
print(f"Number of unique values in 'risks_html' for successful projects: {num_unique_risks_successful}")

Number of unique values in 'risks_html' for failed projects: 22438
Number of unique values in 'risks_html' for successful projects: 22276


In [ ]:
# Function to clean risks_html column based on analysis report
def clean_risks_html_by_report(clean_df, excel_report):
    # Read the analysis report
    report_df = pd.read_excel(excel_report)
    
    # Ensure 'id' column exists in both DataFrames
    if 'id' not in clean_df.columns or 'id' not in report_df.columns:
        print("Error: 'id' column not found in one or both DataFrames")
        return clean_df
    
    # Marking truly empty rows with a placeholder string
    truly_empty_ids = report_df[report_df['truly_empty'] == True]['id'].tolist()
    clean_df.loc[clean_df['id'].isin(truly_empty_ids), 'risks_html'] = "Risks content is truly empty (no user-provided risks)"
    print(f"Found {len(truly_empty_ids)} truly empty risks entries")
    print(f"Truly empty risks are marked. Final DataFrame shape: {clean_df.shape}")
    return clean_df

In [ ]:
# Apply function to clean risks_html based on the analysis report
clean_df = clean_risks_html_by_report(
    clean_df, 
    r"C:\Code\risks_262_analysis_report.xlsx"
)

In [ ]:
# Separate truly empty risks
truly_empty_mask = clean_df['risks_html'] == "Risks content is truly empty (no user-provided risks)"
truly_empty_df = clean_df[truly_empty_mask]
print(f"Number of truly empty risks entries: {truly_empty_df.shape[0]}")

# Drop duplicates from non-empty risks
non_empty_df = clean_df[~truly_empty_mask].drop_duplicates(subset=['risks_html'], keep='first')
print(f"Number of non-empty risks entries after dropping duplicates: {non_empty_df.shape[0]}")

# Concatenate the non-empty and truly empty DataFrames
final_df = pd.concat([non_empty_df, truly_empty_df], ignore_index=True)

print(f"Final DataFrame shape: {final_df.shape}")


In [ ]:
# Check for duplicates in the non-empty risks_html entries
print(clean_df[~truly_empty_mask]['risks_html'].duplicated().sum())

In [14]:
# Save to CSV
final_df.to_csv(r"C:\Code\kickstarter_scrape\clean_df_no_dup_risks2.csv", index=False)